# 🤖 Scores de classification Fink/LSST

Ce notebook explore les **scores et valeurs ajoutées** produits par les modules de science Fink pour les alertes LSST.

Les colonnes Fink sont organisées en 4 familles (préfixes) :
- `f:xm.*` — cross-matches avec catalogues externes (SIMBAD, Gaia DR3, Legacy Survey, VSX...)
- `f:clf.*` — scores de classifieurs ML (binaires ou multi-classes)
- `f:lc_features.<band>.*` — 26 features de courbe de lumière par filtre (SNAD)
- `f:misc.*` — flags, fits statistiques, informations agrégées

Pour un tag donné, ce notebook :
1. Découvre automatiquement les colonnes `f:` disponibles
2. Affiche les distributions des scores de classification (`f:clf.*`)
3. Produit une heatmap objet × score pour identifier les meilleurs candidats
4. Explore les cross-matches (`f:xm.*`)
5. Affiche un tableau récapitulatif trié par score

**API :** `https://api.lsst.fink-portal.org`  
Endpoints : `/api/v1/tags` · `/api/v1/sources` · `/api/v1/schema`

## 1. Paramètres

In [ ]:
# ─── Paramètres utilisateur ───────────────────────────────────────────────────
TAG        = "sn_near_galaxy_candidate"   # Tag à analyser
N_ALERTS   = 50                           # Nombre d'alertes à récupérer
STARTDATE  = None
STOPDATE   = None
BASE_URL   = "https://api.lsst.fink-portal.org"

# Score principal utilisé pour trier le tableau récapitulatif
# None = détecté automatiquement (premier clf disponible)
SORT_SCORE = None

SAVE_FIGS  = True
OUTPUT_DIR = "scores_outputs"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import os
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from IPython.display import display as ipy_display, HTML
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)

mpl.rcParams.update({
    'font.size'        : 11,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'legend.fontsize'  : 9,
    'figure.titlesize' : 13,
    'figure.titleweight': 'bold',
})

MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')

print("Imports OK ✓")

## 3. Découverte du schéma — colonnes `f:` disponibles

On interroge `/api/v1/schema` pour connaître **toutes** les colonnes disponibles,
puis on filtre sur le préfixe `f:` (valeurs ajoutées par Fink).

In [ ]:
resp_schema = requests.post(
    f"{BASE_URL}/api/v1/schema",
    json={"endpoint": "/api/v1/sources"},
    timeout=60
)
resp_schema.raise_for_status()
_schema = resp_schema.json()

# /api/v1/schema retourne un dict à deux sections :
#   "LSST original fields (r:)" : champs originaux LSST  (préfixe r:)
#   "Fink added values (f:)"    : valeurs ajoutées Fink   (préfixe f:)
# Les clés de chaque section sont les noms de colonnes SANS préfixe.

_fink_section = next(
    (v for k, v in _schema.items() if 'fink' in k.lower() or k.startswith('f:')),
    {}
)

# Reconstruction des noms complets f:xxx et catégorisation
def fink_category(col_name):
    c = col_name.replace('f:', '')
    if c.startswith('clf'):         return 'clf'
    if c.startswith('xm'):          return 'xm'
    if c.startswith('lc_features'): return 'lc_features'
    if c.startswith('misc'):        return 'misc'
    return 'other'

# df_fink avec colonnes 'name', 'category', 'type', 'doc'
rows = []
for k, v in _fink_section.items():
    full_name = f"f:{k}"
    rows.append({
        'name'    : full_name,
        'category': fink_category(full_name),
        'type'    : str(v.get('type', '')) if isinstance(v, dict) else '',
        'doc'     : v.get('doc', '')       if isinstance(v, dict) else '',
    })
df_fink = pd.DataFrame(rows)

print(f"Colonnes Fink disponibles : {len(df_fink)}")
for cat, grp in df_fink.groupby('category'):
    print(f"  {cat:15s} : {len(grp):3d} colonnes")

# Listes par catégorie
CLF_COLS  = df_fink[df_fink['category'] == 'clf' ]['name'].tolist()
XM_COLS   = df_fink[df_fink['category'] == 'xm'  ]['name'].tolist()
MISC_COLS = df_fink[df_fink['category'] == 'misc' ]['name'].tolist()

print(f"\nScores clf  : {CLF_COLS}")
print(f"Cross-match : {XM_COLS[:10]}{'...' if len(XM_COLS) > 10 else ''}")
print(f"Misc        : {MISC_COLS}")


## 4. Récupération des alertes avec tous les scores `f:`

In [ ]:
def fetch_tag_alerts(tag, n, startdate=None, stopdate=None):
    payload = {"tag": tag, "n": n, "output-format": "json"}
    if startdate: payload["startdate"] = startdate
    if stopdate:  payload["stopdate"]  = stopdate
    resp = requests.post(f"{BASE_URL}/api/v1/tags", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        raise ValueError(f"Aucune alerte pour '{tag}'.")
    return pd.DataFrame(data)


def fetch_sources_with_scores(dia_object_id, clf_cols):
    """
    Récupère pour un objet ses données photométriques + scores clf.
    """
    base_cols  = "r:diaObjectId,r:diaSourceId,r:midpointMjdTai,r:band,r:psfFlux,r:ra,r:dec"
    extra_cols = ','.join(clf_cols) if clf_cols else ''
    all_cols   = base_cols + (',' + extra_cols if extra_cols else '')

    payload = {
        "diaObjectId"  : str(dia_object_id),
        "columns"      : all_cols,
        "output-format": "json",
    }
    resp = requests.post(f"{BASE_URL}/api/v1/sources", json=payload, timeout=60)

    if resp.status_code != 200:
        return pd.DataFrame()
    try:
        data = resp.json()
    except Exception:
        return pd.DataFrame()
    if not data or isinstance(data, dict):
        return pd.DataFrame()

    df = pd.DataFrame(data)
    df.columns = [c.replace('r:', '').replace('f:', '') for c in df.columns]
    # errors='ignore' est supprimé dans pandas >= 2.0, utiliser 'coerce'
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        except Exception:
            pass  # colonnes non-numériques (band, etc.) restent telles quelles
    return df


# ── Étape 1 : récupérer les diaObjectIds via /tags ────────────────────────────
print(f"Récupération des alertes '{TAG}'...")
df_tags = fetch_tag_alerts(TAG, N_ALERTS, STARTDATE, STOPDATE)
id_col  = [c for c in df_tags.columns if 'diaObjectId' in c][0]
object_ids = df_tags[id_col].astype(str).unique().tolist()
print(f"✓ {len(object_ids)} objet(s) unique(s)\n")

# ── Étape 2 : récupérer les sources + scores clf pour chaque objet ────────────
all_sources = []
n_skip = 0
for oid in tqdm(object_ids, desc='Récupération scores', unit='obj'):
    try:
        df_src = fetch_sources_with_scores(oid, CLF_COLS)
        if not df_src.empty:
            df_src['diaObjectId'] = oid
            all_sources.append(df_src)
        else:
            n_skip += 1
    except Exception as e:
        print(f"  ✗ {oid} — {type(e).__name__}: {e}")
        n_skip += 1

if n_skip:
    print(f"  ⚠️  {n_skip} objet(s) ignoré(s) (inconnus ou sans données)")

if all_sources:
    df_all = pd.concat(all_sources, ignore_index=True)
    print(f"\n✓ {len(df_all)} sources chargées pour {df_all['diaObjectId'].nunique()} objets.")
else:
    df_all = pd.DataFrame()
    print("\n⚠️  Aucune source récupérée.")


In [ ]:
# Diagnostic : réponse brute pour un objet qui échoue
test_fail_id = "313998535398260744"  # un des objets qui plante
base_cols = "r:diaObjectId,r:diaSourceId,r:midpointMjdTai,r:band,r:psfFlux,r:ra,r:dec"
cols_test = base_cols + "," + ",".join(CLF_COLS)

r = requests.post(f"{BASE_URL}/api/v1/sources",
    json={"diaObjectId": test_fail_id, "columns": cols_test, "output-format": "json"},
    timeout=60)

print("status_code :", r.status_code)
print("body brut   :", r.text[:500])
print("type json   :", type(r.json()))
print("json        :", r.json())

## 5. Distribution des scores de classification (`clf.*`)

In [ ]:
# Colonnes clf présentes dans df_all
clf_short = [c.replace('clf.', '') for c in CLF_COLS]
clf_present = [c.replace('f:', '') for c in CLF_COLS if c.replace('f:', '') in df_all.columns]

if not clf_present:
    print("⚠️  Aucune colonne clf disponible dans les données récupérées.")
else:
    n_clf = len(clf_present)
    fig, axes = plt.subplots(1, n_clf, figsize=(5 * n_clf, 4), layout='constrained')
    if n_clf == 1:
        axes = [axes]
    fig.suptitle(f"Distribution des scores de classification — {TAG}")

    for ax, col in zip(axes, clf_present):
        vals = pd.to_numeric(df_all[col], errors='coerce').dropna()
        if vals.empty:
            ax.text(0.5, 0.5, 'Pas de données', ha='center', va='center',
                    transform=ax.transAxes, color='gray')
        else:
            # Histogram avec seuil 0.5 marqué
            n_high = (vals >= 0.5).sum()
            n_low  = (vals <  0.5).sum()
            ax.hist(vals[vals < 0.5],  bins=20, range=(0,1),
                    color='#4477AA', alpha=0.8, label=f'< 0.5 ({n_low})')
            ax.hist(vals[vals >= 0.5], bins=20, range=(0,1),
                    color='#EE6677', alpha=0.8, label=f'≥ 0.5 ({n_high})')
            ax.axvline(0.5, color='black', lw=1.2, ls='--', alpha=0.7)
            ax.set_xlim(0, 1)
            ax.set_xlabel('Score')
            ax.set_ylabel('N alertes')
            ax.legend(fontsize=8)
        short = col.replace('clf.', '')
        ax.set_title(short, fontsize=10)
        ax.grid(True, axis='y', alpha=0.25, linewidth=0.5)

    if SAVE_FIGS:
        fig.savefig(f"{OUTPUT_DIR}/scores_clf_hist_{TAG}.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{OUTPUT_DIR}/scores_clf_hist_{TAG}.png", bbox_inches='tight', dpi=150)
        print(f"✓ Sauvegardé")
    plt.show()

## 6. Heatmap objet × score

Score **médian par objet** (sur toutes ses alertes) — permet d'identifier
les objets les plus probables selon chaque classifieur.

In [ ]:
if not clf_present:
    print("⚠️  Aucun score clf disponible.")
else:
    # Score médian par objet
    df_med = df_all.groupby('diaObjectId')[clf_present].median()
    df_med = df_med.dropna(how='all')

    if SORT_SCORE is None:
        sort_col = clf_present[0]
    else:
        sort_col = SORT_SCORE if SORT_SCORE in df_med.columns else clf_present[0]

    df_med = df_med.sort_values(sort_col, ascending=False)

    n_obj = len(df_med)
    n_clf = len(clf_present)
    fig_h = max(4, n_obj * 0.35)
    fig_w = max(6, n_clf * 2.5)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), layout='constrained')
    fig.suptitle(f"Score médian par objet — {TAG}\n(trié par {sort_col.replace('clf.','')})",
                 fontsize=12)

    im = ax.imshow(df_med.values, aspect='auto', cmap='RdYlGn',
                   vmin=0, vmax=1, origin='upper')

    # Axes
    ax.set_xticks(range(n_clf))
    ax.set_xticklabels([c.replace('clf.','') for c in clf_present],
                        rotation=25, ha='right', fontsize=9)
    ax.set_yticks(range(n_obj))
    ax.set_yticklabels(
        [f"{oid[:10]}..." if len(str(oid)) > 12 else str(oid)
         for oid in df_med.index],
        fontsize=7
    )

    # Valeurs dans les cases
    for i in range(n_obj):
        for j in range(n_clf):
            val = df_med.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}",
                        ha='center', va='center', fontsize=6,
                        color='black' if 0.2 < val < 0.8 else 'white')

    fig.colorbar(im, ax=ax, label='Score médian', shrink=0.6)

    if SAVE_FIGS:
        fig.savefig(f"{OUTPUT_DIR}/scores_heatmap_{TAG}.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{OUTPUT_DIR}/scores_heatmap_{TAG}.png", bbox_inches='tight', dpi=150)
        print(f"✓ Sauvegardé")
    plt.show()

## 7. Cross-matches (`xm.*`) — résultats des catalogues externes

In [ ]:
xm_short  = [c.replace('f:', '') for c in XM_COLS[:5]]
xm_present = [c for c in xm_short if c in df_all.columns]

if not xm_present:
    print("⚠️  Aucune colonne xm disponible dans les données récupérées.")
else:
    print(f"Colonnes cross-match disponibles : {xm_present}\n")

    for col in xm_present:
        vals = df_all[col].dropna()
        if vals.empty:
            print(f"  {col} : aucune valeur")
            continue
        # Si valeurs textuelles (SIMBAD type, TNS name...)
        if vals.dtype == object:
            vc = vals.value_counts().head(15)
            fig, ax = plt.subplots(figsize=(8, 3), layout='constrained')
            ax.barh(vc.index[::-1], vc.values[::-1],
                    color='steelblue', edgecolor='white', linewidth=0.4)
            ax.set_xlabel('N alertes')
            ax.set_title(f"Cross-match {col} — top {len(vc)} valeurs")
            ax.grid(True, axis='x', alpha=0.25, linewidth=0.5)
            if SAVE_FIGS:
                fig.savefig(f"{OUTPUT_DIR}/xm_{col.replace('/','_')}_{TAG}.png",
                            bbox_inches='tight', dpi=150)
            plt.show()
        else:
            # Valeurs numériques (distance, magnitude...)
            fig, ax = plt.subplots(figsize=(6, 3), layout='constrained')
            ax.hist(vals, bins=30, color='steelblue',
                    edgecolor='white', linewidth=0.4, alpha=0.85)
            ax.set_xlabel(col)
            ax.set_ylabel('N alertes')
            ax.set_title(f"Distribution {col}")
            ax.grid(True, axis='y', alpha=0.25, linewidth=0.5)
            if SAVE_FIGS:
                fig.savefig(f"{OUTPUT_DIR}/xm_{col.replace('/','_')}_{TAG}.png",
                            bbox_inches='tight', dpi=150)
            plt.show()

## 8. Tableau récapitulatif — objets triés par score

In [ ]:
# Score médian + infos de base par objet
agg = {
    'psfFlux'       : 'max',
    'midpointMjdTai': 'max',
}
if 'band' in df_all.columns:
    agg['band'] = lambda x: ', '.join(sorted(x.dropna().unique()))

for col in clf_present:
    agg[col] = 'median'

df_summary = df_all.groupby('diaObjectId').agg(agg).reset_index()
df_summary.columns = [
    c.replace('clf.','score_').replace('psfFlux','flux_max (nJy)')
     .replace('midpointMjdTai','last_MJD')
    for c in df_summary.columns
]

# Date lisible
if 'last_MJD' in df_summary.columns:
    df_summary['last_date'] = df_summary['last_MJD'].apply(
        lambda m: (MJD_EPOCH + pd.to_timedelta(float(m), unit='D')).strftime('%Y-%m-%d')
        if pd.notna(m) else '—'
    )
    df_summary = df_summary.drop(columns=['last_MJD'])

# Tri par premier score disponible
score_cols = [c for c in df_summary.columns if c.startswith('score_')]
if score_cols:
    df_summary = df_summary.sort_values(score_cols[0], ascending=False)

# Arrondi
for col in score_cols:
    df_summary[col] = df_summary[col].round(3)
if 'flux_max (nJy)' in df_summary.columns:
    df_summary['flux_max (nJy)'] = df_summary['flux_max (nJy)'].round(1)

# Lien portail
df_summary['Portail Fink'] = df_summary['diaObjectId'].apply(
    lambda oid: f'<a href="https://lsst.fink-portal.org/{oid}" target="_blank">🔗 {oid}</a>'
)

html = df_summary.to_html(index=False, escape=False, border=0, classes='fink-table')
style = """
<style>
  .fink-table { border-collapse: collapse; font-size: 12px; width: 100%; }
  .fink-table th { background: #f0f0f0; padding: 6px 10px;
                   border-bottom: 2px solid #ccc; text-align: left; }
  .fink-table td { padding: 4px 10px; border-bottom: 1px solid #eee;
                   text-align: left; white-space: nowrap; }
  .fink-table tr:hover td { background: #fafafa; }
</style>
"""
ipy_display(HTML(style + html))

## 9. Évolution temporelle des scores pour les meilleurs candidats

Pour les **N objets avec le score le plus élevé**, on trace l'évolution
du score alerte par alerte — utile pour voir si la classification
se stabilise ou varie au cours du temps.

In [ ]:
N_TOP = 5   # Nombre de meilleurs candidats à afficher

if not clf_present or df_summary.empty:
    print("⚠️  Pas de scores disponibles.")
else:
    top_ids = df_summary.head(N_TOP)['diaObjectId'].tolist()

    for clf_col in clf_present:
        fig, ax = plt.subplots(figsize=(12, 4), layout='constrained')
        short = clf_col.replace('clf.', '')
        fig.suptitle(f"Évolution du score '{short}' — top {N_TOP} candidats")

        colors = plt.cm.tab10(np.linspace(0, 1, N_TOP))

        for idx, oid in enumerate(top_ids):
            sub = df_all[df_all['diaObjectId'] == oid].copy()
            sub['midpointMjdTai'] = pd.to_numeric(sub['midpointMjdTai'], errors='coerce')
            sub = sub.dropna(subset=['midpointMjdTai', clf_col]).sort_values('midpointMjdTai')
            if sub.empty:
                continue
            dates = MJD_EPOCH + pd.to_timedelta(sub['midpointMjdTai'].values, unit='D')
            label = str(oid)[:14] + '...'
            ax.plot(dates, pd.to_numeric(sub[clf_col], errors='coerce'),
                    'o-', color=colors[idx], ms=4, lw=1.2,
                    alpha=0.85, label=label)

        ax.axhline(0.5, color='gray', lw=1.0, ls='--', alpha=0.6, label='Seuil 0.5')
        ax.set_ylim(-0.05, 1.05)
        ax.set_ylabel(f'Score {short}')
        ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
        ax.legend(fontsize=8, ncol=2, loc='lower right')
        ax.grid(True, alpha=0.2, linewidth=0.5)

        if SAVE_FIGS:
            stem = f"{OUTPUT_DIR}/score_evolution_{short}_{TAG}"
            fig.savefig(f"{stem}.pdf", bbox_inches='tight', dpi=150)
            fig.savefig(f"{stem}.png", bbox_inches='tight', dpi=150)
            print(f"✓ Sauvegardé : {stem}")
        plt.show()